# March Mania · Shooting environment and matchup profiles

**Research round 02 — build features, test one historical season, inspect, stop.**

The official submitted reference remains **0.1222672**; **0.1097454** is the historical research target. This notebook does not submit, score 2026, or replace the final model. It tests **14 new candidates** on top of a fixed **16-feature** reference using **eight** 2019 tournament-classifier fits.

The two edited repository notebooks stay untouched. This notebook runs only the standalone modules in this companion kit; it reads the existing raw data and verifies source/data identity. No data downloads, package installation, Git writes, GPUs, or cloud jobs occur.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown, FileLink

KIT = Path(os.environ.get("MARCH_KIT", str(Path.cwd()))).expanduser().resolve()
if not (KIT / "shot_features.py").is_file():
    KIT = Path.home() / "march_shooting_research"
REPO = Path(os.environ.get("MARCH_REPO", str(Path.home() / "march-machine-learning-mania-2026"))).expanduser().resolve()
assert (KIT / "shot_features.py").is_file(), "Open the notebook in the extracted research kit."
assert Path(sys.executable).parent.resolve() == (REPO / ".venv/bin").resolve(), "Select the existing Python (March Mania) kernel."
sys.path.insert(0, str(KIT))
from run_round02 import run_stage
from research_plots import figures
pio.renderers.default = "plotly_mimetype"
print("Kit:", KIT)
print("Repository (read only):", REPO)
print("Kernel:", sys.executable)
print("Run the supplied synthetic safety tests before this notebook.")

## 1 · Build the new representations

**Question R:** Did opponents shoot unusually well or poorly relative to their own other games, after excluding all same-opponent meetings? Six residual/schedule features test this idea. Residuals are not identified causal luck.

**Question P:** Do separate, opportunity-weighted opponent adjustments for 2P accuracy, 3P accuracy and 3PA share expose useful matchup information? Six rating effects and two cross-team compositions test this idea.

The first stage builds 14 season/population snapshots (2013–2019, men and women). Each uses five small regular-season rating regressions. It does not read tournament targets. Fixed feature definitions, cutoffs, limitations and primary sources are in **RESEARCH_PLAN.md**.

Preparation has a **600-second hard cap**. Completed snapshots have content-verified receipts and are reused.

In [ ]:
run_stage("prepare", phase="smoke", max_seconds=600)

In [ ]:
latest = json.loads((KIT / "reports/latest_run.json").read_text())
RUN = Path(latest["run_dir"])
registry = pd.read_csv(RUN / "feature_registry.csv")
coverage = pd.DataFrame([json.loads(p.read_text()) for p in sorted((RUN / "snapshots").glob("*/coverage.json"))])
display(Markdown("### Built feature catalog"))
display(registry[["feature", "family", "new_candidate", "swap_parity"]])
display(Markdown("### Seeded-team raw-data support"))
display(coverage)
print("Fingerprint:", latest["fingerprint"])
print("No feature benefit is established by successful construction alone.")

## 2 · Fixed comparison, not an algorithm sweep

| Recipe | Features | New information |
|---|---:|---|
| anchor | 16 | Existing strength, efficiency and shooting concepts |
| anchor_residual | 22 | Add six opponent-excluded shooting residual/schedule differences |
| anchor_profile | 24 | Add eight adjusted shot-profile/matchup features |
| anchor_both | 30 | Add both families |

Each recipe uses the same logistic C=0.1, training-only scaling and support checks. There is no capacity-based selection or retuning. Men and women train separately on 2013–2018 and validate on 2019 main-draw games. Mirroring is only for training; each validation game is scored once.

This is a small feature diagnostic, not a reproduction of the best pooled-XGBoost / women's-conference submission. The historical season is already consumed development evidence. **A gain is not a new Kaggle score.**

In [ ]:
run_stage("evaluate", phase="smoke", max_seconds=600)

In [ ]:
metrics = pd.read_csv(RUN / "smoke_metrics.csv")
aggregate = pd.read_csv(RUN / "smoke_aggregate.csv")
ablations = pd.read_csv(RUN / "smoke_ablations.csv")
display(metrics[["Gender", "Season", "recipe", "train_games", "games", "brier", "delta_vs_anchor", "active_features", "swap_error"]])
display(aggregate)
display(ablations)
print("Negative delta favors adding the feature family. One season is not a promotion decision.")

## 3 · Inspect the representation and the predictions

The charts separate construction, model performance, conditional feature contributions and calibration. A coefficient is not a causal effect. Plotly hover details expose team support, probabilities and fold comparisons.

In [ ]:
plots = figures(RUN, "smoke")
for fig in plots:
    fig.show()
print(f"Displayed {len(plots)} interactive figures.")

## 4 · Checkpoint and report

Rendering never retrains. The HTML contains its own Plotly JavaScript and can be opened offline. The return ZIP contains only the run metadata, feature registry and aggregate/fold evidence, not your raw inputs, existing notebooks or model files.

In [ ]:
run_stage("report", phase="smoke", max_seconds=120)
display(FileLink(str((RUN / "smoke_report.html").relative_to(KIT))))
display(FileLink("reports/milestone_02_return.zip"))

In [ ]:
summary = json.loads((RUN / "smoke_summary.json").read_text())
display(pd.DataFrame({"check": ["Status", "New classifier fits in this attempt", "Reused fits", "New candidates", "Official score changed", "GitHub updated", "All challengers clearly worse"],
                     "value": [summary["status"], summary["new_tournament_classifier_fits"], summary["reused_tournament_classifier_fits"], summary["new_matchup_candidates"], False, summary["github_updated"], summary["clearly_unproductive_smoke"]]}))
print("Save this notebook and return reports/milestone_02_return.zip. Stop this milestone here.")

## Decision before more compute

A technical pass means the features and experiment ran, not that a family works. Inspect support, Brier deltas, calibration and possible failure patterns. If every challenger is worse by more than 0.01 for both populations, the larger panel is blocked for diagnosis.

Otherwise, the next deliberate milestone is the unchanged five-season comparison in notebook 03, followed by fixed-reference family ablations and stability assessment. **Do not automatically start notebook 03 now.** Feature engineering remains open.